# Mid Fusion Lite - Kaggle Run All
Notebook nay duoc thiet ke de bam Run All la train duoc.

## Ban can gan 2 dataset trong Kaggle Notebook
1. Dataset code cua ban (chua folder `mid_fusion_lite`).
2. Dataset ISIC 2024 chua cac file metadata va HDF5 (`train-metadata.csv`, `test-metadata.csv`, `sample_submission.csv`, `train-image.hdf5`, `test-image.hdf5`).

## Quan trong
Chi can sua duong dan o Cell 2 (phan USER CONFIG).

In [ ]:
# =========================
# USER CONFIG (CHI SUA CELL NAY)
# =========================

from pathlib import Path

# Preset: 'smoke' (nhanh de test), 'quick' (can bang), 'full_lite' (nang hon).
# Fair-compare voi TABM standalone: nen dung full_lite.
PRESET = 'full_lite'

# Neu muon chi train 1 vai fold de test nhanh:
#   '0' hoac '0,1'
# Neu muon train tat ca fold cua preset: None
FOLDS = None

# Co the de None de dung mac dinh cua preset.
# Fair-compare: de epochs cao hon quick/smoke.
EPOCHS_OVERRIDE = 20
IMAGE_SIZE_OVERRIDE = 160

# Trong moi truong kaggle khong internet, de False de tranh loi tai pretrained.
# Khi internet on va muon dung pretrained weights, set True.
USE_PRETRAINED_BACKBONE = False

# Neu ban co file weights backbone local trong kaggle dataset, dien duong dan o day.
# Vi du: '/kaggle/input/my-backbone-weights/efficientnet_b0_backbone.pth'
BACKBONE_WEIGHTS_PATH = r'D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite\convnextv2_nano_backbone.pth'
BACKBONE_NAME_OVERRIDE = 'convnextv2_nano'

# ===== Fair-compare overrides (co the de None de dung preset) =====
TRAIN_IMAGE_BACKBONE_OVERRIDE = True
USE_TABM_TABULAR_OVERRIDE = True
USE_QUANTILE_TRANSFORM_OVERRIDE = True
QUANTILE_NOISE_STD_OVERRIDE = 1e-5
FUSION_MODE_OVERRIDE = 'cross_attention'
CROSS_ATTENTION_HEADS_OVERRIDE = 4
CROSS_ATTENTION_DROPOUT_OVERRIDE = 0.1

TABM_ARCH_TYPE_OVERRIDE = 'tabm'
TABM_K_OVERRIDE = 16
TABM_N_BLOCKS_OVERRIDE = 3
TABM_D_BLOCK_OVERRIDE = 256
TAB_DROPOUT_OVERRIDE = 0.15

LR_OVERRIDE = 3e-4
WEIGHT_DECAY_OVERRIDE = 2e-4
PATIENCE_OVERRIDE = 5
NEGATIVE_RATIO_OVERRIDE = 10
TRAIN_BATCH_SIZE_OVERRIDE = 16
VALID_BATCH_SIZE_OVERRIDE = 32
SHOW_PREDICT_PROGRESS_OVERRIDE = True
PREDICT_LOG_INTERVAL_OVERRIDE = 20
NUM_WORKERS_OVERRIDE = None

# Multi-GPU (DataParallel) settings
USE_DATA_PARALLEL_OVERRIDE = False
DATA_PARALLEL_DEVICE_IDS_OVERRIDE = None

# Dat None de khong cat bot train/valid step (de danh gia cong bang hon)
MAX_TRAIN_STEPS_OVERRIDE = None
MAX_VALID_STEPS_OVERRIDE = None

# Feature engineering memory switch:
#   None  -> dung theo config mac dinh
#   False -> tat FE nang (goi y cho local RAM thap)
#   True  -> bat FE nang
USE_FEATURE_ENGINEERING_OVERRIDE = False

# ===== Advanced tabular feature toggles =====
# Nhom feature context (khuyen nghi bat)
USE_CONTEXTUAL_GROUP_NORM_FEATURES_OVERRIDE = False
USE_UGLY_DUCKLING_FEATURES_OVERRIDE = False
USE_ANATOMICAL_ZSCORE_FEATURES_OVERRIDE = False
USE_AREA_AGGREGATION_FEATURES_OVERRIDE = False

# Nhom nang: co the cham/ton RAM hon
USE_PATIENT_LOF_FEATURES_OVERRIDE = False
PATIENT_LOF_MIN_GROUP_SIZE_OVERRIDE = 5

USE_NULL_IMPORTANCE_FILTER_OVERRIDE = False
NULL_IMPORTANCE_REPEATS_OVERRIDE = 8
NULL_IMPORTANCE_QUANTILE_OVERRIDE = 0.95
NULL_IMPORTANCE_MIN_KEPT_OVERRIDE = 64

# ===== Run mode: 'auto' | 'kaggle' | 'local' =====
RUN_MODE = 'auto'

# Kaggle defaults
KAGGLE_CODE_DATASET_ROOT = '/kaggle/input/datasets/khoangdz/isic-2024'
KAGGLE_COMP_DATASET_ROOT = '/kaggle/input/competitions/isic-2024-challenge'
KAGGLE_OUTPUT_DIR = '/kaggle/working/mid_fusion_lite_outputs'

# Local overrides (de None de auto-detect theo workspace)
LOCAL_CODE_DATASET_ROOT = r"D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite"
LOCAL_DATA_ROOT = r"D:\HK1_2025_2026\SEMINAR_CODE\data"
LOCAL_OUTPUT_DIR = './local_outputs'


def _first_existing(candidates):
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return p
    return None


if RUN_MODE not in {'auto', 'kaggle', 'local'}:
    raise ValueError("RUN_MODE must be one of: auto, kaggle, local")

if RUN_MODE == 'auto':
    is_kaggle = Path('/kaggle/input').exists()
else:
    is_kaggle = RUN_MODE == 'kaggle'

# Resolve code root robustly for both kaggle and local
if is_kaggle:
    code_root = Path(KAGGLE_CODE_DATASET_ROOT)
else:
    if LOCAL_CODE_DATASET_ROOT is not None and str(LOCAL_CODE_DATASET_ROOT).strip() != '':
        code_root = Path(str(LOCAL_CODE_DATASET_ROOT))
    else:
        code_root = _first_existing([
            './mid_fusion_lite',
            '.',
            '../mid_fusion_lite',
        ])
        if code_root is None:
            code_root = Path('./mid_fusion_lite')

CODE_DATASET_ROOT = str(code_root)

if is_kaggle:
    comp_root = Path(KAGGLE_COMP_DATASET_ROOT)
    TRAIN_METADATA_PATH = str(comp_root / 'train-metadata.csv')
    TEST_METADATA_PATH = str(comp_root / 'test-metadata.csv')
    SAMPLE_SUBMISSION_PATH = str(comp_root / 'sample_submission.csv')
    TRAIN_HDF5_PATH = str(comp_root / 'train-image.hdf5')
    TEST_HDF5_PATH = str(comp_root / 'test-image.hdf5')
    OUTPUT_DIR = KAGGLE_OUTPUT_DIR
else:
    if LOCAL_DATA_ROOT is not None and str(LOCAL_DATA_ROOT).strip() != '':
        local_root = Path(str(LOCAL_DATA_ROOT))
        TRAIN_METADATA_PATH = str(local_root / 'train-metadata.csv')
        TEST_METADATA_PATH = str(local_root / 'test-metadata.csv')
        SAMPLE_SUBMISSION_PATH = str(local_root / 'sample_submission.csv')
        TRAIN_HDF5_PATH = str(local_root / 'train-image.hdf5')
        TEST_HDF5_PATH = str(local_root / 'test-image.hdf5')
    else:
        train_meta = _first_existing(['./data/train-metadata.csv', '../data/train-metadata.csv'])
        test_meta = _first_existing(['./tests/test-metadata.csv', '../tests/test-metadata.csv'])
        sample_sub = _first_existing(['./tests/sample_submission.csv', '../tests/sample_submission.csv'])
        train_h5 = _first_existing(['./data/train-image.hdf5', '../data/train-image.hdf5'])
        test_h5 = _first_existing(['./tests/test-image.hdf5', '../tests/test-image.hdf5'])

        TRAIN_METADATA_PATH = str(train_meta if train_meta is not None else Path('./data/train-metadata.csv'))
        TEST_METADATA_PATH = str(test_meta if test_meta is not None else Path('./tests/test-metadata.csv'))
        SAMPLE_SUBMISSION_PATH = str(sample_sub if sample_sub is not None else Path('./tests/sample_submission.csv'))
        TRAIN_HDF5_PATH = str(train_h5 if train_h5 is not None else Path('./data/train-image.hdf5'))
        TEST_HDF5_PATH = str(test_h5 if test_h5 is not None else Path('./tests/test-image.hdf5'))

    OUTPUT_DIR = LOCAL_OUTPUT_DIR

# Auto worker policy: local safe by default, kaggle faster by default
if NUM_WORKERS_OVERRIDE is None:
    NUM_WORKERS_OVERRIDE = 4 if is_kaggle else 0

print('Environment mode:', 'KAGGLE' if is_kaggle else 'LOCAL')
print('CODE_DATASET_ROOT:', CODE_DATASET_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('NUM_WORKERS_OVERRIDE:', NUM_WORKERS_OVERRIDE)

Environment mode: LOCAL
CODE_DATASET_ROOT: D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite
OUTPUT_DIR: ./local_outputs
NUM_WORKERS_OVERRIDE: 0


In [2]:
from pathlib import Path
import os

if Path('/kaggle/input').exists():
    print('Kaggle input datasets:')
    print(os.listdir('/kaggle/input'))
else:
    print('Kaggle input not found. Running in local mode.')

paths_to_check = {
    'CODE_DATASET_ROOT': CODE_DATASET_ROOT,
    'TRAIN_METADATA_PATH': TRAIN_METADATA_PATH,
    'TEST_METADATA_PATH': TEST_METADATA_PATH,
    'SAMPLE_SUBMISSION_PATH': SAMPLE_SUBMISSION_PATH,
    'TRAIN_HDF5_PATH': TRAIN_HDF5_PATH,
    'TEST_HDF5_PATH': TEST_HDF5_PATH,
}

missing = []
for key, value in paths_to_check.items():
    exists = Path(value).exists()
    print(f'{key}: {value} | exists={exists}')
    if not exists:
        missing.append(key)

runner_path = Path(CODE_DATASET_ROOT) / 'runner.py'
runner_exists = runner_path.exists()
print(f'RUNNER_PATH: {runner_path} | exists={runner_exists}')
if not runner_exists:
    missing.append('CODE_DATASET_ROOT/runner.py')

if missing:
    raise FileNotFoundError(
        'Thieu duong dan: ' + ', '.join(missing) +
        '. Hay sua lai CELL 2 cho dung ten dataset/path.'
    )

print('Path check OK.')

Kaggle input not found. Running in local mode.
CODE_DATASET_ROOT: D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite | exists=True
TRAIN_METADATA_PATH: D:\HK1_2025_2026\SEMINAR_CODE\data\train-metadata.csv | exists=True
TEST_METADATA_PATH: D:\HK1_2025_2026\SEMINAR_CODE\data\test-metadata.csv | exists=True
SAMPLE_SUBMISSION_PATH: D:\HK1_2025_2026\SEMINAR_CODE\data\sample_submission.csv | exists=True
TRAIN_HDF5_PATH: D:\HK1_2025_2026\SEMINAR_CODE\data\train-image.hdf5 | exists=True
TEST_HDF5_PATH: D:\HK1_2025_2026\SEMINAR_CODE\data\test-image.hdf5 | exists=True
RUNNER_PATH: D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite\runner.py | exists=True
Path check OK.


In [3]:
# Cai dependencies theo requirements trong folder code
!pip -q install -r "{CODE_DATASET_ROOT}/requirements.txt"

In [4]:
import importlib
import sys
from pathlib import Path

if CODE_DATASET_ROOT in sys.path:
    sys.path.remove(CODE_DATASET_ROOT)
sys.path.insert(0, CODE_DATASET_ROOT)

# Ensure notebook uses the latest local files instead of stale modules in kernel state.
import config as config_module
import tabular as tabular_module
import train as train_module
import runner as runner_module

config_module = importlib.reload(config_module)
tabular_module = importlib.reload(tabular_module)
train_module = importlib.reload(train_module)
runner_module = importlib.reload(runner_module)

print('Using runner from:', Path(runner_module.__file__).resolve())
print('Using config from:', Path(config_module.__file__).resolve())
print('Using tabular from:', Path(tabular_module.__file__).resolve())
print('Using train from:', Path(train_module.__file__).resolve())

from runner import run_with_preset

overrides = {
    'output_dir': OUTPUT_DIR,
    'train_metadata_path': TRAIN_METADATA_PATH,
    'test_metadata_path': TEST_METADATA_PATH,
    'sample_submission_path': SAMPLE_SUBMISSION_PATH,
    'train_hdf5_path': TRAIN_HDF5_PATH,
    'test_hdf5_path': TEST_HDF5_PATH,
    'use_pretrained_backbone': bool(USE_PRETRAINED_BACKBONE),
}

if BACKBONE_WEIGHTS_PATH is not None and str(BACKBONE_WEIGHTS_PATH).strip() != '':
    overrides['backbone_weights_path'] = str(BACKBONE_WEIGHTS_PATH)

if FOLDS is not None and str(FOLDS).strip() != '':
    overrides['fold_indices'] = [int(x.strip()) for x in str(FOLDS).split(',') if x.strip()]

if EPOCHS_OVERRIDE is not None:
    overrides['epochs'] = int(EPOCHS_OVERRIDE)

if IMAGE_SIZE_OVERRIDE is not None:
    overrides['image_size'] = int(IMAGE_SIZE_OVERRIDE)

if BACKBONE_NAME_OVERRIDE is not None and str(BACKBONE_NAME_OVERRIDE).strip() != '':
    overrides['backbone_name'] = str(BACKBONE_NAME_OVERRIDE).strip()

if TRAIN_IMAGE_BACKBONE_OVERRIDE is not None:
    overrides['train_image_backbone'] = bool(TRAIN_IMAGE_BACKBONE_OVERRIDE)

use_feature_engineering_override = globals().get('USE_FEATURE_ENGINEERING_OVERRIDE', None)
if use_feature_engineering_override is not None:
    overrides['use_feature_engineering'] = bool(use_feature_engineering_override)

if USE_CONTEXTUAL_GROUP_NORM_FEATURES_OVERRIDE is not None:
    overrides['use_contextual_group_norm_features'] = bool(USE_CONTEXTUAL_GROUP_NORM_FEATURES_OVERRIDE)

if USE_UGLY_DUCKLING_FEATURES_OVERRIDE is not None:
    overrides['use_ugly_duckling_features'] = bool(USE_UGLY_DUCKLING_FEATURES_OVERRIDE)

if USE_ANATOMICAL_ZSCORE_FEATURES_OVERRIDE is not None:
    overrides['use_anatomical_zscore_features'] = bool(USE_ANATOMICAL_ZSCORE_FEATURES_OVERRIDE)

if USE_AREA_AGGREGATION_FEATURES_OVERRIDE is not None:
    overrides['use_area_aggregation_features'] = bool(USE_AREA_AGGREGATION_FEATURES_OVERRIDE)

if USE_PATIENT_LOF_FEATURES_OVERRIDE is not None:
    overrides['use_patient_lof_features'] = bool(USE_PATIENT_LOF_FEATURES_OVERRIDE)

if PATIENT_LOF_MIN_GROUP_SIZE_OVERRIDE is not None:
    overrides['patient_lof_min_group_size'] = int(PATIENT_LOF_MIN_GROUP_SIZE_OVERRIDE)

if USE_NULL_IMPORTANCE_FILTER_OVERRIDE is not None:
    overrides['use_null_importance_filter'] = bool(USE_NULL_IMPORTANCE_FILTER_OVERRIDE)

if NULL_IMPORTANCE_REPEATS_OVERRIDE is not None:
    overrides['null_importance_repeats'] = int(NULL_IMPORTANCE_REPEATS_OVERRIDE)

if NULL_IMPORTANCE_QUANTILE_OVERRIDE is not None:
    overrides['null_importance_quantile'] = float(NULL_IMPORTANCE_QUANTILE_OVERRIDE)

if NULL_IMPORTANCE_MIN_KEPT_OVERRIDE is not None:
    overrides['null_importance_min_kept'] = int(NULL_IMPORTANCE_MIN_KEPT_OVERRIDE)

if USE_TABM_TABULAR_OVERRIDE is not None:
    overrides['use_tabm_tabular'] = bool(USE_TABM_TABULAR_OVERRIDE)

if FUSION_MODE_OVERRIDE is not None and str(FUSION_MODE_OVERRIDE).strip() != '':
    overrides['fusion_mode'] = str(FUSION_MODE_OVERRIDE).strip()

if CROSS_ATTENTION_HEADS_OVERRIDE is not None:
    overrides['cross_attention_heads'] = int(CROSS_ATTENTION_HEADS_OVERRIDE)

if CROSS_ATTENTION_DROPOUT_OVERRIDE is not None:
    overrides['cross_attention_dropout'] = float(CROSS_ATTENTION_DROPOUT_OVERRIDE)

if USE_QUANTILE_TRANSFORM_OVERRIDE is not None:
    overrides['use_quantile_transform'] = bool(USE_QUANTILE_TRANSFORM_OVERRIDE)

if QUANTILE_NOISE_STD_OVERRIDE is not None:
    overrides['quantile_noise_std'] = float(QUANTILE_NOISE_STD_OVERRIDE)

if TABM_ARCH_TYPE_OVERRIDE is not None and str(TABM_ARCH_TYPE_OVERRIDE).strip() != '':
    overrides['tabm_arch_type'] = str(TABM_ARCH_TYPE_OVERRIDE).strip()

if TABM_K_OVERRIDE is not None:
    overrides['tabm_k'] = int(TABM_K_OVERRIDE)

if TABM_N_BLOCKS_OVERRIDE is not None:
    overrides['tabm_n_blocks'] = int(TABM_N_BLOCKS_OVERRIDE)

if TABM_D_BLOCK_OVERRIDE is not None:
    overrides['tabm_d_block'] = int(TABM_D_BLOCK_OVERRIDE)

if TAB_DROPOUT_OVERRIDE is not None:
    overrides['tab_dropout'] = float(TAB_DROPOUT_OVERRIDE)

if LR_OVERRIDE is not None:
    overrides['lr'] = float(LR_OVERRIDE)

if WEIGHT_DECAY_OVERRIDE is not None:
    overrides['weight_decay'] = float(WEIGHT_DECAY_OVERRIDE)

if PATIENCE_OVERRIDE is not None:
    overrides['patience'] = int(PATIENCE_OVERRIDE)

if NEGATIVE_RATIO_OVERRIDE is not None:
    overrides['negative_ratio'] = int(NEGATIVE_RATIO_OVERRIDE)

if NUM_WORKERS_OVERRIDE is not None:
    overrides['num_workers'] = int(NUM_WORKERS_OVERRIDE)

if TRAIN_BATCH_SIZE_OVERRIDE is not None:
    overrides['train_batch_size'] = int(TRAIN_BATCH_SIZE_OVERRIDE)

if VALID_BATCH_SIZE_OVERRIDE is not None:
    overrides['valid_batch_size'] = int(VALID_BATCH_SIZE_OVERRIDE)

if SHOW_PREDICT_PROGRESS_OVERRIDE is not None:
    overrides['show_predict_progress'] = bool(SHOW_PREDICT_PROGRESS_OVERRIDE)

if PREDICT_LOG_INTERVAL_OVERRIDE is not None:
    overrides['predict_log_interval'] = int(PREDICT_LOG_INTERVAL_OVERRIDE)

if USE_DATA_PARALLEL_OVERRIDE is not None:
    overrides['use_data_parallel'] = bool(USE_DATA_PARALLEL_OVERRIDE)

if DATA_PARALLEL_DEVICE_IDS_OVERRIDE is not None:
    overrides['data_parallel_device_ids'] = [
        int(x) for x in DATA_PARALLEL_DEVICE_IDS_OVERRIDE
    ]

if MAX_TRAIN_STEPS_OVERRIDE is not None:
    overrides['max_train_steps_per_epoch'] = int(MAX_TRAIN_STEPS_OVERRIDE)

if MAX_VALID_STEPS_OVERRIDE is not None:
    overrides['max_valid_steps'] = int(MAX_VALID_STEPS_OVERRIDE)

print('Run config:')
print({'preset': PRESET, **overrides})

result = run_with_preset(PRESET, **overrides)

c:\Users\Khoadz\.conda\envs\Final_Project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using runner from: D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite\runner.py
Using config from: D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite\config.py
Using tabular from: D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite\tabular.py
Using train from: D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite\train.py
Run config:
{'preset': 'full_lite', 'output_dir': './local_outputs', 'train_metadata_path': 'D:\\HK1_2025_2026\\SEMINAR_CODE\\data\\train-metadata.csv', 'test_metadata_path': 'D:\\HK1_2025_2026\\SEMINAR_CODE\\data\\test-metadata.csv', 'sample_submission_path': 'D:\\HK1_2025_2026\\SEMINAR_CODE\\data\\sample_submission.csv', 'train_hdf5_path': 'D:\\HK1_2025_2026\\SEMINAR_CODE\\data\\train-image.hdf5', 'test_hdf5_path': 'D:\\HK1_2025_2026\\SEMINAR_CODE\\data\\test-image.hdf5', 'use_pretrained_backbone': False, 'backbone_weights_path': 'D:\\HK1_2025_2026\\SEMINAR_CODE\\mid_fusion_lite\\convnextv2_nano_backbone.pth', 'epochs': 20, 'image_size': 160, 'backbone_name': 'convnextv2_nano', 'train_imag

c:\Users\Khoadz\.conda\envs\Final_Project\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Loaded local backbone weights: D:\HK1_2025_2026\SEMINAR_CODE\mid_fusion_lite\convnextv2_nano_backbone.pth | missing=0 | unexpected=0
Epoch 01 | train_loss=15.08994 | valid_loss=1.71437 | valid_auroc=0.92525 | valid_pauc=0.14339


KeyboardInterrupt: 

In [ ]:
import json
import pandas as pd
import shutil
from pathlib import Path

print('CV pAUC:', result['cv_pauc'])
print('OOF file:', result['oof_path'])
print('Submission file:', result['submission_path'])
print('Metrics file:', result['metrics_path'])

metrics = json.loads(Path(result['metrics_path']).read_text())
print('Fold metrics:')
for row in metrics.get('fold_metrics', []):
    print(row)

# Luu them mot file submission.csv o /kaggle/working de gui len competition nhanh
shutil.copy(result['submission_path'], '/kaggle/working/submission.csv')
print('Copied to /kaggle/working/submission.csv')

display(pd.read_csv('/kaggle/working/submission.csv').head())